# PipelineWatch-NG — Module 5: Weekly Near Real-Time Update
**Run this notebook every 7 days to keep the dashboard current.**

---

## How to use

1. Open this notebook once a week
2. **Kernel → Restart and Run All**
3. Wait ~5 minutes
4. Dashboard at pipelinewatch-ng.streamlit.app updates automatically

## What it does

- Pulls the last 7 days of FIRMS/VIIRS fire detections (3-6 hour latency)
- Pulls the last 7 days of TROPOMI SO₂ (3-5 hour latency)
- Compares against the pre-computed 2023 baseline
- Raises HIGH / MEDIUM / LOW alert based on anomaly detection
- Appends to a running history CSV
- Commits and pushes to GitHub (Streamlit auto-redeploys)

## Alert levels

| Level | Condition | Recommended action |
|-------|-----------|-------------------|
| HIGH | Fire anomaly AND SO₂ anomaly both triggered | NNPC field inspection |
| MEDIUM | One signal anomalous | Continue weekly monitoring |
| LOW | All clear | No action required |

---

## Outputs

| File | Description |
|------|-------------|
| data/cached/m5_nrt_latest.json | Most recent alert report |
| data/cached/m5_nrt_history.csv | Running weekly history log |
| outputs/m5_nrt_trend.html | Interactive trend chart (2+ weeks) |


## Cell 1 — Setup

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — see Module 1 notes
import matplotlib.pyplot as plt
from IPython.display import IFrame, display

import ee
import os
import json
import gc
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
import xgboost as xgb
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler, MinMaxScaler

GEE_PROJECT_ID = 'pipelinewatch-ng'
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print('GEE connected: ' + str(ee.Number(42).getInfo()))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

CACHE_DIR  = '../data/cached'
MODEL_DIR  = '../data/models'
OUTPUT_DIR = '../outputs'

# Auto-calculate the last 7 days
TODAY     = datetime.utcnow()
NRT_END   = TODAY.strftime('%Y-%m-%d')
NRT_START = (TODAY - timedelta(days=7)).strftime('%Y-%m-%d')

print('NRT window: ' + NRT_START + ' to ' + NRT_END)


## Cell 2 — Config and load models

In [ ]:
ROI_BOUNDS = [6.50, 5.00, 7.20, 5.80]
ROI = ee.Geometry.Rectangle(ROI_BOUNDS)

BASELINE_START     = '2023-01-01'
BASELINE_END       = '2023-06-30'
FIRMS_BRIGHTNESS_K = 330.0
SO2_THRESHOLD_DU   = 1.5    # sensitive for NRT

# Load trained models from Module 3
model_path  = os.path.join(MODEL_DIR, 'xgb_risk_scorer.json')
scaler_path = os.path.join(MODEL_DIR, 'feature_scaler.pkl')

if os.path.exists(model_path):
    xgb_model  = xgb.XGBClassifier()
    xgb_model.load_model(model_path)
    scaler     = joblib.load(scaler_path)
    with open(os.path.join(CACHE_DIR, 'm3_model_config.json')) as f:
        model_cfg = json.load(f)
    FEATURE_COLS = model_cfg['feature_cols']
    print('Models loaded. Features: ' + str(FEATURE_COLS))
else:
    print('WARNING: models not found — run Module 3 first')
    FEATURE_COLS = []


## Cell 3 — Fetch NRT FIRMS (last 7 days)

In [ ]:
print('=== NRT FIRMS/VIIRS (' + NRT_START + ' to ' + NRT_END + ') ===')

firms_nrt = (
    ee.ImageCollection('FIRMS')
    .filterDate(NRT_START, NRT_END).filterBounds(ROI)
    .select(['T21', 'confidence', 'line_number'])
)
n_firms_nrt = firms_nrt.size().getInfo()
print('FIRMS images this week: ' + str(n_firms_nrt))

if n_firms_nrt > 0:
    firms_nrt_comp = (
        firms_nrt.select('T21').max().rename('T21_max').clip(ROI)
        .addBands(firms_nrt.select('T21').count().rename('fire_count').clip(ROI))
    )
    gc.collect()
    hot_px = (
        firms_nrt_comp.select('T21_max').gt(FIRMS_BRIGHTNESS_K)
        .reduceRegion(reducer=ee.Reducer.sum(), geometry=ROI,
                      scale=375, maxPixels=1e9, bestEffort=True).getInfo()
    )
    n_nrt_fire_pixels = int(hot_px.get('T21_max', 0) or 0)
    # Baseline weekly average: ~50 hotspots over 26 weeks
    baseline_weekly_avg = 50 / 26
    anomaly_flag = n_nrt_fire_pixels > (baseline_weekly_avg * 1.5)
    print('Hot pixels this week: ' + str(n_nrt_fire_pixels))
    print('Baseline weekly avg : ' + str(round(baseline_weekly_avg, 1)))
    if anomaly_flag:
        print('>> ALERT: Fire activity above baseline this week')
else:
    print('No FIRMS data this week (possible NRT coverage gap)')
    n_nrt_fire_pixels = 0
    anomaly_flag = False


## Cell 4 — Fetch NRT TROPOMI SO₂ (last 7 days)

In [ ]:
print('=== NRT TROPOMI SO2 (' + NRT_START + ' to ' + NRT_END + ') ===')

tropomi_nrt = (
    ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_SO2')
    .filterDate(NRT_START, NRT_END).filterBounds(ROI)
    .select(['SO2_column_number_density', 'cloud_fraction'])
)
n_tropomi_nrt = tropomi_nrt.size().getInfo()
print('TROPOMI images this week: ' + str(n_tropomi_nrt))

if n_tropomi_nrt > 0:
    def mask_and_convert(image):
        cloud  = image.select('cloud_fraction')
        so2_du = image.select('SO2_column_number_density').multiply(2241.5).rename('SO2_DU')
        return so2_du.updateMask(cloud.lt(0.4))
    clean_nrt = tropomi_nrt.map(mask_and_convert)
    so2_nrt   = clean_nrt.mean().rename('SO2_mean_DU').clip(ROI)

    gc.collect()
    so2_stats = so2_nrt.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
        geometry=ROI, scale=5500, maxPixels=1e9, bestEffort=True).getInfo()

    so2_mean_nrt = so2_stats.get('SO2_mean_DU_mean', 0) or 0
    so2_max_nrt  = so2_stats.get('SO2_mean_DU_max',  0) or 0
    so2_anomaly  = so2_mean_nrt > SO2_THRESHOLD_DU

    print('SO2 mean this week  : ' + str(round(so2_mean_nrt, 3)) + ' DU')
    print('SO2 max this week   : ' + str(round(so2_max_nrt,  3)) + ' DU')
    if so2_anomaly:
        print('>> ALERT: SO2 above threshold this week')
else:
    so2_mean_nrt = 0
    so2_max_nrt  = 0
    so2_anomaly  = False
    print('No TROPOMI data this week.')


## Cell 5 — Alert report

In [ ]:
combined_alert = anomaly_flag or so2_anomaly
alert_level    = 'HIGH' if (anomaly_flag and so2_anomaly) else 'MEDIUM' if combined_alert else 'LOW'

print()
print('=' * 55)
print('PIPELINEWATCH-NG  NRT ALERT REPORT')
print('Week of: ' + NRT_START + ' to ' + NRT_END)
print('=' * 55)
print('Alert level          : ' + alert_level)
print('Fire pixels          : ' + str(n_nrt_fire_pixels))
print('Fire anomaly         : ' + str(anomaly_flag))
print('SO2 mean             : ' + str(round(so2_mean_nrt, 3)) + ' DU')
print('SO2 anomaly          : ' + str(so2_anomaly))
print()

if alert_level == 'HIGH':
    print('ACTION: Both fire and SO2 anomalies detected.')
    print('Recommend NNPC field inspection at known HIGH-risk coordinates.')
elif alert_level == 'MEDIUM':
    print('MONITOR: Single signal anomaly. Continue weekly monitoring.')
else:
    print('All clear. No anomalies above threshold this week.')

# Save report
nrt_report = {
    'run_date': datetime.utcnow().isoformat(),
    'nrt_window': NRT_START + ' to ' + NRT_END,
    'alert_level': alert_level,
    'fire_pixels': n_nrt_fire_pixels,
    'fire_anomaly': bool(anomaly_flag),
    'so2_mean_du': round(so2_mean_nrt, 3),
    'so2_anomaly': bool(so2_anomaly),
    'firms_images': n_firms_nrt,
    'tropomi_images': n_tropomi_nrt,
}
with open(os.path.join(CACHE_DIR, 'm5_nrt_latest.json'), 'w') as f:
    json.dump(nrt_report, f, indent=2)
print('Report saved: data/cached/m5_nrt_latest.json')


## Cell 6 — Update history and trend chart

In [ ]:
history_path = os.path.join(CACHE_DIR, 'm5_nrt_history.csv')
new_row = pd.DataFrame([{'date': NRT_END, 'fire_pixels': n_nrt_fire_pixels,
                          'so2_mean_du': round(so2_mean_nrt, 3),
                          'alert_level': alert_level,
                          'fire_anomaly': anomaly_flag, 'so2_anomaly': so2_anomaly}])

if os.path.exists(history_path):
    df_history = pd.read_csv(history_path)
    df_history = pd.concat([df_history, new_row], ignore_index=True)
    df_history = df_history.drop_duplicates(subset=['date']).sort_values('date')
else:
    df_history = new_row

df_history.to_csv(history_path, index=False)
print('History: ' + str(len(df_history)) + ' weekly records')

if len(df_history) >= 2:
    df_history['date'] = pd.to_datetime(df_history['date'])

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('Weekly VIIRS fire pixel count',
                                        'Weekly TROPOMI SO2 mean'))

    fig.add_trace(go.Scatter(
        x=df_history['date'], y=df_history['fire_pixels'],
        mode='lines+markers', name='Fire pixels',
        line=dict(color='#E24B4A', width=2), marker=dict(size=6)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=df_history['date'], y=df_history['so2_mean_du'],
        mode='lines+markers', name='SO2 mean',
        line=dict(color='#1D9E75', width=2), marker=dict(size=6)
    ), row=2, col=1)
    fig.add_hline(y=SO2_THRESHOLD_DU, line_dash='dash', line_color='#854F0B',
                  annotation_text='Threshold', row=2, col=1)

    fig.update_layout(
        title='PipelineWatch-NG — Weekly NRT Trend<br>Trans Niger Pipeline corridor',
        height=520, plot_bgcolor='white', paper_bgcolor='white'
    )
    fig.update_yaxes(title_text='Hot pixels', row=1, col=1, gridcolor='#f0f0f0')
    fig.update_yaxes(title_text='SO2 (DU)',   row=2, col=1, gridcolor='#f0f0f0')

    out = os.path.join(OUTPUT_DIR, 'm5_nrt_trend.html')
    fig.write_html(out)
    print('Trend chart saved: ' + out)
    display(IFrame(src=out, width='100%', height=550))
else:
    print('Run again next week to see trend chart (needs 2+ data points)')


## Cell 7 — Push to GitHub (auto-updates dashboard)

In [ ]:
import subprocess

project_root = os.path.abspath('../')

def run_git(cmd):
    result = subprocess.run(cmd, shell=True, cwd=project_root,
                            capture_output=True, text=True)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    return result.returncode

commit_msg = 'NRT update ' + NRT_END + ' | alert=' + alert_level

run_git('git add data/cached/ outputs/')
run_git('git commit -m "' + commit_msg + '"')
run_git('git push')

print()
print('Done. Dashboard updates within 2 minutes at:')
print('https://pipelinewatch-ng.streamlit.app')
